In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.environ.get("GROQ_API_KEY")

In [2]:
from langchain_groq import ChatGroq

model=ChatGroq(model="openai/gpt-oss-20b",groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.15'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002CC49046900>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002CC490A44A0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
model.invoke([
    HumanMessage(content="Hello, My name is Kajal")
])

AIMessage(content='Hello Kajal! 👋 How’s your day going? Anything in particular you’d like to chat about or need help with?', additional_kwargs={'reasoning_content': 'The user says "Hello, My name is Kajal". We need to respond. It\'s a greeting, we can say hello, ask how they\'re doing. Also maybe ask about their day or how can we help. The user is presumably wanting a conversation. There\'s no conflict with policy. So just greet.'}, response_metadata={'token_usage': {'completion_tokens': 97, 'prompt_tokens': 78, 'total_tokens': 175, 'completion_time': 0.115286367, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.004311889, 'prompt_tokens_details': None, 'queue_time': 0.311640388, 'total_time': 0.119598256}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_996f667773', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a023fe-d78a-76c3-9ee4-25203dec5d14-0', tool_calls=[], invalid_tool_calls

In [6]:
model.invoke([
    HumanMessage(content="Hello, My name is Kajal and I am a Blockchain Developer"),
    AIMessage(content="'Hello Kajal! 👋 How’s your day going? Anything in particular you’d like to chat about or need help with?'"),
    HumanMessage(content="Hey, What's my name and What do i do for a living?"),
])

AIMessage(content='You’re **Kajal**, and you’re a **Blockchain Developer**.', additional_kwargs={'reasoning_content': 'User: "Hey, What\'s my name and What do i do for a living?" We already have name: Kajal. Profession: Blockchain Developer. So answer: your name is Kajal, you are a blockchain developer. Probably short.'}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 135, 'total_tokens': 208, 'completion_time': 0.087225268, 'completion_tokens_details': {'reasoning_tokens': 49}, 'prompt_time': 0.006823087, 'prompt_tokens_details': None, 'queue_time': 0.290984961, 'total_time': 0.094048355}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_4a35f7bd1b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a02405-a8ee-7ca0-91b7-8c43c7403868-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 135, 'output_tokens': 73, 'total_tokens': 208, 'output_token_details': {'re

## Message History

We can use a message history class to wrap our model and make it stateful.
This wll keep track of inputs and outputs of the model, and store them in some datastore.
Future interactions will then load messages and pass them into the chain as part of the input.

In [9]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)

c:\Users\kajal\OneDrive\Dokumen\Langchain\venev\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [10]:
config={"configurable":{"session_id":"chat1"}}

In [ ]:
response=with_message_history.invoke(
    [HumanMessage(content="Hello, My name is Kajal and I am a Blockchain Developer")],
     config=config
)

In [15]:
response.content

'Hey Kajal! 👋 It’s awesome to connect with someone who’s into blockchain development. What’s on your radar today? Are you building a dApp, diving into a new protocol, or looking for some best‑practice tips? Let me know how I can help!'

In [17]:
with_message_history.invoke([HumanMessage(content="Hey, What's my name and What do i do for a living?")], config=config).content

'Your name is **Kajal**, and you’re a **Blockchain Developer**—building smart contracts, decentralized applications, and working on blockchain protocols.'

In [18]:
## change the config ===> session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hello, What is my name")],
     config=config1
)
response.content

'I’m not sure what your name is—could you let me know?'